## Step 1: Install packages
Notice the extra `langchain*` packages — these are the wrappers.

In [1]:
!pip install -q langchain-text-splitters langchain-community langchain-huggingface langchain-core sentence-transformers faiss-cpu transformers accelerate


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Same raw text as before

In [2]:
raw_text = """
RAG, or Retrieval-Augmented Generation, is a technique that combines information retrieval with text generation. Instead of relying only on what a language model learned during training, RAG lets the model pull in relevant external documents at query time, then use those documents as context to generate an answer.

The core idea behind RAG is simple: a language model's internal knowledge is frozen at training time and can become outdated or incomplete. By retrieving fresh, relevant text from an external knowledge base, the model can answer questions about recent events, private company data, or niche topics it was never trained on.

A typical RAG pipeline has two phases. The indexing phase happens once: documents are collected, parsed into clean text, split into chunks, converted into vector embeddings, and stored in a vector database. The query phase happens every time a user asks something: the query is embedded, the vector database is searched for the most similar chunks, and those chunks are inserted into a prompt sent to the language model.

Chunking strategy matters a lot in RAG systems. If chunks are too large, irrelevant information gets mixed in with relevant information, diluting the context. If chunks are too small, important context might get split across multiple chunks and lose coherence. Common approaches use a fixed token or word count per chunk with some overlap between consecutive chunks so information near chunk boundaries is not lost.

Embeddings are numerical vector representations of text where semantically similar pieces of text end up close together in vector space. Embedding models are trained so that sentences with similar meaning produce vectors with a small distance between them, even if the exact wording is very different. This is what allows semantic search to work, as opposed to simple keyword matching.

FAISS, developed by Meta AI, is a library for efficient similarity search over large sets of vectors. It can be used entirely locally without any external database service, which makes it a common choice for prototyping and learning how vector search works before moving to a production database like Pinecone, Weaviate, or Chroma.

There is a family of more advanced RAG variants beyond the naive pipeline. Advanced RAG adds steps like query rewriting and re-ranking of retrieved chunks. Modular RAG treats the pipeline as swappable components and can include hybrid search, routing between multiple knowledge sources, iterative retrieval, and agentic behavior where the model decides when and what to retrieve.
"""
print(f"Loaded {len(raw_text.split())} words")

Loaded 404 words


## Step 3: Chunking — LangChain's TextSplitter instead of your own function

Raw version: your `chunk_text()` function, splitting by word count.
LangChain version: `RecursiveCharacterTextSplitter` — same idea (chunk_size + overlap), prebuilt, and smarter about not cutting mid-sentence when possible.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,      # characters, not words this time
    chunk_overlap=80,
)
chunks = splitter.split_text(raw_text)
print(f"Created {len(chunks)} chunks")
print("\nFirst chunk:\n", chunks[0])

c:\Users\acer\OneDrive\Desktop\RAG Model\LangChain_Model\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Created 9 chunks

First chunk:
 RAG, or Retrieval-Augmented Generation, is a technique that combines information retrieval with text generation. Instead of relying only on what a language model learned during training, RAG lets the model pull in relevant external documents at query time, then use those documents as context to generate an answer.


## Step 4 + 5: Embedding + FAISS store — LangChain wraps BOTH into one class

Raw version: you called `SentenceTransformer(...)` yourself, then manually built a `faiss.IndexFlatL2`.
LangChain version: `HuggingFaceEmbeddings` wraps the embedding model, and `FAISS.from_texts` wraps chunking storage AND embedding into a single call — you never touch a raw FAISS index directly.

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_texts(chunks, embeddings_model)
print("Vector store built")

C:\Users\acer\AppData\Local\Temp\ipykernel_3796\3445439132.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
c:\Users\acer\OneDrive\Desktop\RAG Model\LangChain_Model\env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\acer\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.


Vector store built


## Step 6: Turn the vector store into a retriever

Raw version: you wrote a `retrieve()` function that manually called `index.search()`.
LangChain version: `.as_retriever()` does that for you — one line.

In [5]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# quick manual test of just the retriever, before wiring up the LLM
test_results = retriever.invoke("What is chunking and why does chunk size matter?")
for i, doc in enumerate(test_results, 1):
    print(f"[{i}] {doc.page_content[:150]}...\n")

[1] chunk with some overlap between consecutive chunks so information near chunk boundaries is not lost....

[2] Chunking strategy matters a lot in RAG systems. If chunks are too large, irrelevant information gets mixed in with relevant information, diluting the ...

[3] for the most similar chunks, and those chunks are inserted into a prompt sent to the language model....



## Step 7: Load the LLM — wrapped in LangChain's interface

Raw version: you called `transformers.pipeline(...)` directly.
LangChain version: same underlying pipeline, wrapped in `HuggingFacePipeline` so LangChain's chain-building syntax can call it.

In [6]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

hf_pipeline = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", max_new_tokens=150)
llm = HuggingFacePipeline(pipeline=hf_pipeline)
print("LLM loaded")

c:\Users\acer\OneDrive\Desktop\RAG Model\LangChain_Model\env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\acer\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 311.21it/s]
[transformers]

LLM loaded


## Step 8: Build the prompt template

Raw version: you wrote `build_prompt()` — an f-string.
LangChain version: `PromptTemplate` — same idea, formalized into a class so it plugs into chains.

In [7]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    template="Answer using only this context.\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:",
    input_variables=["context", "question"],
)

## Step 9: Chain it all together — this is the piece with no raw equivalent

This is the part your raw `ask()` function did manually: retrieve -> build prompt -> call LLM -> return answer.
LangChain calls this a "chain." The `|` operator pipes one step's output into the next step's input.

In [8]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)
print("Chain built")

Chain built


## Step 10: Ask a question — one line now

In [9]:
query = "What is chunking and why does chunk size matter?"
result = rag_chain.invoke(query)
# result includes the echoed prompt since this model isn't instruction-tuned for clean output via this pipeline path
answer = result[len(prompt_template.format(context=format_docs(retriever.invoke(query)), question=query)):].strip()
print("=== ANSWER ===")
print(answer)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


=== ANSWER ===
Chunking is an approach used in RAG (Reactive Auto-generative Language Generation) systems where chunks of text are generated from a given input sentence. The goal of chunking is to preserve the contextual meaning of the original text by grouping related words together. This helps ensure that the system can generate coherent sentences based on the provided input. 

In RAG systems, chunking is crucial because if chunks are too large, irrelevant information gets mixed in with relevant information, which can lead to a loss of context. Smaller chunks may also be prone to losing important context as they are spread out over multiple chunks, potentially resulting in disorganized output. Therefore, choosing appropriate chunk sizes is essential for maintaining the integrity of the generated text. Additionally, having


In [12]:
query = "What is chunking and why does chunk size matter?"

result = rag_chain.invoke(query)
echoed_prompt = prompt_template.format(
    context=format_docs(retriever.invoke(query)),
    question=query
)
answer = result[len(echoed_prompt):].strip()

print("=== ANSWER ===")
print(answer)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== ANSWER ===
Chunking is an approach used in natural language processing (NLP) to divide long documents or text into smaller, more manageable pieces called chunks. The goal of chunking is to ensure that the information within each chunk is relevant and coherent, rather than being scattered throughout the document.
The impact of chunk size on chunking can be significant. Smaller chunks may contain less information but will be easier for the language model to process. However, if the chunk size is too small, it may lead to irrelevant information getting mixed in with relevant information, which could result in loss of context.
In summary, chunking helps maintain the coherence and relevance of the text by ensuring that the information within each chunk is meaningful and aligned with the overall context of the document


## Compare side by side

| Raw version | LangChain version |
|---|---|
| `chunk_text()` your own function | `RecursiveCharacterTextSplitter` |
| `SentenceTransformer` + manual `faiss.IndexFlatL2` | `HuggingFaceEmbeddings` + `FAISS.from_texts` |
| `retrieve()` calling `index.search()` | `.as_retriever()` |
| `transformers.pipeline()` called directly | `HuggingFacePipeline` wrapping the same pipeline |
| f-string in `build_prompt()` | `PromptTemplate` |
| `ask()` manually gluing every step | `rag_chain` built with `|` operators |

Same 10 steps, same underlying libraries (sentence-transformers, FAISS, transformers) doing the actual work.
LangChain just gives every step a standard class and lets you glue them with `|` instead of writing the glue code yourself.